# 9. Hyperparameter Tuning

## 1. Introduction

Hyperparameter tuning is used to identify model configurations
that improve predictive performance.

The selected model from the model-building stage will be optimized
using RandomizedSearchCV.

The optimization metric is Average Precision.

The test dataset will not be used during hyperparameter tuning.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

from src.config import (
    CLEANED_DATA_PATH,
    TARGET_COLUMN,
    RANDOM_STATE,
    TEST_SIZE,
)

df = pd.read_csv(CLEANED_DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN])

y = df[TARGET_COLUMN]

X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.20,
    stratify=y_development,
    random_state=RANDOM_STATE
)

print("Training Shape:", X_train.shape)

print("Validation Shape:", X_validation.shape)

Training Shape: (182276, 30)
Validation Shape: (45569, 30)


## 2. Model Configuration

The model and sampling strategy should be selected based
on the validation experiments performed in the previous notebook.

For this example, Random Forest with SMOTE is used.

If the best experiment from Notebook 08 is different,
update the configuration below accordingly.

In [2]:
MODEL_NAME = "random_forest"

SAMPLING_METHOD = "smote"

print("Model:", MODEL_NAME)

print("Sampling:", SAMPLING_METHOD)

Model: random_forest
Sampling: smote


## 3. Randomized Search Cross-Validation

RandomizedSearchCV evaluates randomly selected combinations
from a predefined hyperparameter search space.

StratifiedKFold maintains class proportions across folds.

The complete imbalanced-learn Pipeline is passed to the
cross-validation procedure.

Consequently, preprocessing and sampling are fitted separately
within each training fold.

In [3]:
from src.tuning import (
    tune_model,
    save_tuned_model,
)

search = tune_model(
    X_train,
    y_train,
    model_name=MODEL_NAME,
    sampling_method=SAMPLING_METHOD,
    n_iter=20,
    cv_splits=5,
    scoring="average_precision"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [4]:
print("Best Parameters:")

for parameter, value in search.best_params_.items():

    print(f"{parameter}: {value}")

print("\nBest Cross-Validation Average Precision:")

print(search.best_score_)

Best Parameters:
model__n_estimators: 500
model__min_samples_split: 2
model__min_samples_leaf: 4
model__max_depth: None

Best Cross-Validation Average Precision:
0.8511758864315537


## 4. Validation Evaluation

The best estimator is evaluated on the independent validation set.

This step provides an estimate of performance on unseen
development data.

The validation set is also used to select a classification
threshold according to the project's evaluation objective.

In [5]:
from src.evaluate import (
    get_prediction_probabilities,
    calculate_metrics,
)

best_model = search.best_estimator_

y_validation_prob = get_prediction_probabilities(
    best_model,
    X_validation
)

validation_metrics = calculate_metrics(
    y_validation,
    y_validation_prob,
    threshold=0.50
)

for metric_name, metric_value in validation_metrics.items():

    print(
        f"{metric_name}: {metric_value:.4f}"
    )

threshold: 0.5000
accuracy: 0.9994
precision: 0.8732
recall: 0.7848
f1_score: 0.8267
roc_auc: 0.9730
average_precision: 0.8060
mcc: 0.8276
balanced_accuracy: 0.8923


## 5. Classification Threshold Selection

The default classification threshold is 0.50.

However, fraud detection may require a different threshold.

A threshold will be selected using the validation set.

For this experiment, the threshold that maximizes F1-Score
will be selected.

This threshold must be fixed before evaluating the final
test dataset.

In [6]:
from src.evaluate import (
    choose_threshold,
    save_threshold,
)

selected_threshold = choose_threshold(
    y_validation,
    y_validation_prob,
    metric="f1"
)

print("Selected Threshold:", selected_threshold)

save_threshold(
    selected_threshold
)

print("Threshold saved successfully.")

Selected Threshold: 0.6522238095238094
Threshold saved successfully.


In [7]:
threshold_results = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, selected_threshold]:

    metrics = calculate_metrics(
        y_validation,
        y_validation_prob,
        threshold=threshold
    )

    threshold_results.append(metrics)

threshold_comparison = pd.DataFrame(
    threshold_results
).drop_duplicates(
    subset=["threshold"]
)

display(
    threshold_comparison.sort_values(
        by="threshold"
    )
)

,threshold,accuracy,precision,recall,f1_score,roc_auc,average_precision,mcc,balanced_accuracy
0,0.100000,0.995501,0.257692,0.848101,0.395280,0.972983,0.806035,0.466082,0.921929
1,0.200000,0.998530,0.550000,0.835443,0.663317,0.972983,0.806035,0.677202,0.917128
2,0.300000,0.999166,0.730337,0.822785,0.773810,0.972983,0.806035,0.774772,0.911129
3,0.400000,0.999342,0.818182,0.797468,0.807692,0.972983,0.806035,0.807429,0.898580
4,0.500000,0.999429,0.873239,0.784810,0.826667,0.972983,0.806035,0.827563,0.892306
5,0.652224,0.999451,0.909091,0.759494,0.827586,0.972983,0.806035,0.830669,0.879681


## 6. Saving the Tuned Model

The best estimator is saved as a complete pipeline.

The saved pipeline includes:

- Feature Engineering
- Preprocessing
- Sampling configuration
- Trained Classification Model

The classification threshold is stored separately.

The saved artifacts will be used during final evaluation
and prediction.

In [8]:
from src.config import TUNED_MODEL_PATH

save_tuned_model(
    search,
    file_path=TUNED_MODEL_PATH
)

print("Tuned model saved successfully.")

Tuned model saved to: D:\Python-Workspace\Projects\Data-Science\Credit-Card-Fraud-Detection\models\best_model_tuned.joblib
Tuned model saved successfully.


In [9]:
import json

from src.config import MODEL_METADATA_PATH

metadata = {
    "model_name": MODEL_NAME,
    "sampling_method": SAMPLING_METHOD,
    "scoring_metric": "average_precision",
    "best_cv_score": float(search.best_score_),
    "selected_threshold": float(selected_threshold),
    "random_state": RANDOM_STATE,
    "features": X.columns.tolist(),
}

with open(
    MODEL_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4
    )

print("Model metadata saved.")

Model metadata saved.


## 7. Conclusion

Hyperparameter tuning was performed using stratified
cross-validation.

The best model was selected based on Average Precision.

The final classification threshold was selected using
validation data only.

The tuned model and its metadata have been saved for
final test evaluation.